In [1]:
!git clone https://github.com/heyuepeng/PFLlibVSP.git


fatal: destination path 'PFLlibVSP' already exists and is not an empty directory.


In [2]:
! pip install calmsize

In [3]:
! pip install opacus


In [4]:
!pip install numpy==1.23.5
!pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 113.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89990 sha256=2773d84cadfe4f5b0297722870e22f0dc0a5f1d149bfa2f21f16399c7d6630f0
  Stored in directory: /root/.cache/pip/wheels/fc/5d/45/34fe9945d5e45e261134e72284395be36c2d4828af38e2b0fe
Successfully built lit
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
  Attempting uninstall: torch
    Found existing installation: torch 2.6.0+cu124
    Uninstalling torch-2.6.0+cu124:
      Successfully uninstalled torch-2.6.0+

In [5]:
pip install fvcore


In [13]:
# PFLlib: Personalized Federated Learning Algorithm Library
# Copyright (C) 2021  Jianqing Zhang

# This program is free software; you can redistribute it and/or modify
# it under the terms of the GNU General Public License as published by
# the Free Software Foundation; either version 2 of the License, or
# (at your option) any later version.

# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU General Public License for more details.

# You should have received a copy of the GNU General Public License along
# with this program; if not, write to the Free Software Foundation, Inc.,
# 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA.

#!/usr/bin/env python

import copy
import torch
import argparse
import os
import time
import warnings
import numpy as np
import torchvision
import logging
import psutil


import sys
sys.path.append('/content/PFLlibVSP/system')

from fvcore.nn import FlopCountAnalysis

from flcore.servers.serveravg import FedAvg
from flcore.servers.serverpFedMe import pFedMe
from flcore.servers.serverperavg import PerAvg
from flcore.servers.serverprox import FedProx
from flcore.servers.serverfomo import FedFomo
from flcore.servers.serveramp import FedAMP
from flcore.servers.servermtl import FedMTL
from flcore.servers.serverlocal import Local
from flcore.servers.serverper import FedPer
from flcore.servers.serverapfl import APFL
from flcore.servers.serverditto import Ditto
from flcore.servers.serverrep import FedRep
from flcore.servers.serverphp import FedPHP
from flcore.servers.serverbn import FedBN
from flcore.servers.serverrod import FedROD
from flcore.servers.serverproto import FedProto
from flcore.servers.serverdyn import FedDyn
from flcore.servers.servermoon import MOON
from flcore.servers.serverbabu import FedBABU
from flcore.servers.serverapple import APPLE
from flcore.servers.servergen import FedGen
from flcore.servers.serverscaffold import SCAFFOLD
from flcore.servers.serverdistill import FedDistill
from flcore.servers.serverala import FedALA
from flcore.servers.serverawa import FedAWA
from flcore.servers.serverpac import FedPAC
from flcore.servers.serverlg import LG_FedAvg
from flcore.servers.servergc import FedGC
from flcore.servers.serverfml import FML
from flcore.servers.serverkd import FedKD
from flcore.servers.serverpcl import FedPCL
from flcore.servers.servercp import FedCP
from flcore.servers.servergpfl import GPFL
from flcore.servers.serverntd import FedNTD
from flcore.servers.servergh import FedGH
from flcore.servers.serveravgDBE import FedAvgDBE

from flcore.trainmodel.models import *

from flcore.trainmodel.bilstm import *
from flcore.trainmodel.resnet import *
from flcore.trainmodel.alexnet import *
from flcore.trainmodel.mobilenet_v2 import *
from flcore.trainmodel.transformer import *

from utils.result_utils import average_data
from utils.mem_utils import MemReporter

import sys
sys.argv = ['']


logger = logging.getLogger()
logger.setLevel(logging.ERROR)

log_filename = "run_summary.txt"
summary_log = open(log_filename, "w")

def log_both(text):
    print(text)
    summary_log.write(str(text) + "\n")


warnings.simplefilter("ignore")
torch.manual_seed(0)

# hyper-params for Text tasks
vocab_size = 98635   #98635 for AG_News and 399198 for Sogou_News
max_len=200
emb_dim=32

def run(args, log_both):

    time_list = []
    reporter = MemReporter()
    model_str = args.model

    for i in range(args.prev, args.times):
        print(f"\n============= Running time: {i}th =============")
        log_both("Creating server and clients ...")
        start = time.time()

        # Generate args.model
        if model_str == "mlr": # convex
            if "mnist" in args.dataset:
                args.model = Mclr_Logistic(1*28*28, num_classes=args.num_classes).to(args.device)
            elif "Cifar10" in args.dataset:
                args.model = Mclr_Logistic(3*32*32, num_classes=args.num_classes).to(args.device)
            else:
                args.model = Mclr_Logistic(60, num_classes=args.num_classes).to(args.device)

        elif model_str == "cnn": # non-convex
            if "mnist" in args.dataset:
                args.model = FedAvgCNN(in_features=1, num_classes=args.num_classes, dim=1024).to(args.device)
            elif "Cifar10" in args.dataset:
                args.model = FedAvgCNN(in_features=3, num_classes=args.num_classes, dim=1600).to(args.device)
            elif "omniglot" in args.dataset:
                args.model = FedAvgCNN(in_features=1, num_classes=args.num_classes, dim=33856).to(args.device)
                # args.model = CifarNet(num_classes=args.num_classes).to(args.device)
            elif "Digit5" in args.dataset:
                args.model = Digit5CNN().to(args.device)
            else:
                args.model = FedAvgCNN(in_features=3, num_classes=args.num_classes, dim=10816).to(args.device)

        elif model_str == "dnn": # non-convex
            if "mnist" in args.dataset:
                args.model = DNN(1*28*28, 100, num_classes=args.num_classes).to(args.device)
            elif "Cifar10" in args.dataset:
                args.model = DNN(3*32*32, 100, num_classes=args.num_classes).to(args.device)
            else:
                args.model = DNN(60, 20, num_classes=args.num_classes).to(args.device)

        elif model_str == "resnet":
            args.model = torchvision.models.resnet18(pretrained=False, num_classes=args.num_classes).to(args.device)

            # args.model = torchvision.models.resnet18(pretrained=True).to(args.device)
            # feature_dim = list(args.model.fc.parameters())[0].shape[1]
            # args.model.fc = nn.Linear(feature_dim, args.num_classes).to(args.device)

            # args.model = resnet18(num_classes=args.num_classes, has_bn=True, bn_block_num=4).to(args.device)

        elif model_str == "resnet10":
            args.model = resnet10(num_classes=args.num_classes).to(args.device)

        elif model_str == "resnet34":
            args.model = torchvision.models.resnet34(pretrained=False, num_classes=args.num_classes).to(args.device)

        elif model_str == "alexnet":
            args.model = alexnet(pretrained=False, num_classes=args.num_classes).to(args.device)

            # args.model = alexnet(pretrained=True).to(args.device)
            # feature_dim = list(args.model.fc.parameters())[0].shape[1]
            # args.model.fc = nn.Linear(feature_dim, args.num_classes).to(args.device)

        elif model_str == "googlenet":
            args.model = torchvision.models.googlenet(pretrained=False, aux_logits=False, num_classes=args.num_classes).to(args.device)

            # args.model = torchvision.models.googlenet(pretrained=True, aux_logits=False).to(args.device)
            # feature_dim = list(args.model.fc.parameters())[0].shape[1]
            # args.model.fc = nn.Linear(feature_dim, args.num_classes).to(args.device)

        elif model_str == "mobilenet_v2":
            args.model = mobilenet_v2(pretrained=False, num_classes=args.num_classes).to(args.device)

            # args.model = mobilenet_v2(pretrained=True).to(args.device)
            # feature_dim = list(args.model.fc.parameters())[0].shape[1]
            # args.model.fc = nn.Linear(feature_dim, args.num_classes).to(args.device)
        # elif model_str == "mutisslstm":
        #     if "driving" == args.dataset:
        #         args.model = MultiHead_Seq2SeqLSTM(input_dim=12, hidden_dim=128, decoder_output_dim=1, num_layers=2,
        #                                  future_seq_length=5, dropout=0.04).to(args.device)
        elif model_str == "sslstm":
            if "driving" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=12, hidden_dim=128, decoder_output_dim=1, num_layers=2,
                                         future_seq_length=5, dropout=0.1).to(args.device)
            elif "driving_fg2" == args.dataset or "driving_fg3" == args.dataset or "driving_fg4" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=15, hidden_dim=128, decoder_output_dim=1, num_layers=2,
                                         future_seq_length=5, dropout=0.1).to(args.device)
            elif "driving_fg24" == args.dataset or "driving_fg34" == args.dataset or "driving_fg34_jr" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=18, hidden_dim=128, decoder_output_dim=1, num_layers=2,
                                         future_seq_length=5, dropout=0.1).to(args.device)
            elif "driving_fg234" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=21, hidden_dim=128, decoder_output_dim=1, num_layers=2,
                                         future_seq_length=5, dropout=0.1).to(args.device)
            elif "driving10" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=17, hidden_dim=128, decoder_output_dim=1, num_layers=3,
                                         future_seq_length=10, dropout=0.2).to(args.device)
            elif "driving10_fg2" == args.dataset or "driving10_fg3" == args.dataset or "driving10_fg4" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=20, hidden_dim=128, decoder_output_dim=1, num_layers=3,
                                         future_seq_length=10, dropout=0.2).to(args.device)
            elif "driving10_fg24" == args.dataset or "driving10_fg34" == args.dataset or "driving10_fg34_jr" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=23, hidden_dim=128, decoder_output_dim=1, num_layers=3,
                                         future_seq_length=10, dropout=0.2).to(args.device)
            elif "driving10_fg234" == args.dataset:
                args.model = MultiHead_Seq2SeqLSTM(input_dim=26, hidden_dim=128, decoder_output_dim=1, num_layers=3,
                                         future_seq_length=10, dropout=0.2).to(args.device)
        elif model_str == "lstm":
            if "driving" == args.dataset:
                args.model = DrivingLSTM(input_dim=12, hidden_dim=128, output_dim=5, num_layers=2, bidirectional=False,
                                         dropout=0.1).to(args.device)
            elif "driving_fg24" == args.dataset or "driving_fg34" == args.dataset or "driving_fg34_jr" == args.dataset:
                args.model = DrivingLSTM(input_dim=18, hidden_dim=128, output_dim=5, num_layers=2, bidirectional=False,
                                         dropout=0.1).to(args.device)
            elif "driving10" == args.dataset:
                args.model = DrivingLSTM(input_dim=17, hidden_dim=128, output_dim=10, num_layers=2, bidirectional=False,
                                         dropout=0.1).to(args.device)
            elif "driving10_fg24" == args.dataset or "driving10_fg34" == args.dataset or "driving10_fg34_jr" == args.dataset:
                args.model = DrivingLSTM(input_dim=23, hidden_dim=128, output_dim=10, num_layers=2, bidirectional=False,
                                         dropout=0.1).to(args.device)
            else:
                args.model = LSTMNet(hidden_dim=emb_dim, vocab_size=vocab_size, num_classes=args.num_classes).to(args.device)
        elif model_str == "informer":
            if "driving" in args.dataset:
                args.model = Informer(
		        enc_in=23,
		        dec_in=23,
		        c_out=1,
		        seq_len=10,
		        label_len=5,
		        out_len=5,       # ← this line fixes your current crash
		        d_model=128,
		        n_heads=4,
		        e_layers=2,
		        d_layers=1,
		        dropout=0.1
		    ).to(args.device)
        elif model_str == "bilstm":
            args.model = BiLSTM_TextClassification(input_size=vocab_size, hidden_size=emb_dim, output_size=args.num_classes,
                        num_layers=1, embedding_dropout=0, lstm_dropout=0, attention_dropout=0,
                        embedding_length=emb_dim).to(args.device)

        elif model_str == "fastText":
            args.model = fastText(hidden_dim=emb_dim, vocab_size=vocab_size, num_classes=args.num_classes).to(args.device)

        elif model_str == "TextCNN":
            args.model = TextCNN(hidden_dim=emb_dim, max_len=max_len, vocab_size=vocab_size,
                            num_classes=args.num_classes).to(args.device)

        elif model_str == "Transformer":
            if "driving" in args.dataset:
                args.model = SimplifiedDrivingTransformer(input_dim=12, d_model=128, nhead=4, num_encoder_layers=3,
                    dim_feedforward=512, dropout=0.1, output_dim=5).to(args.device)
            else:
                args.model = TransformerModel(ntoken=vocab_size, d_model=emb_dim, nhead=8, d_hid=emb_dim, nlayers=2,
                            num_classes=args.num_classes).to(args.device)

        elif model_str == "AmazonMLP":
            args.model = AmazonMLP().to(args.device)

        elif model_str == "harcnn":
            if args.dataset == 'har':
                args.model = HARCNN(9, dim_hidden=1664, num_classes=args.num_classes, conv_kernel_size=(1, 9), pool_kernel_size=(1, 2)).to(args.device)
            elif args.dataset == 'pamap':
                args.model = HARCNN(9, dim_hidden=3712, num_classes=args.num_classes, conv_kernel_size=(1, 9), pool_kernel_size=(1, 2)).to(args.device)

        else:
            raise NotImplementedError

        log_both(str(args.model))

        # select algorithm
        if args.algorithm == "FedAvg":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedAvg(args, i)

        elif args.algorithm == "Local":
            server = Local(args, i)

        elif args.algorithm == "FedMTL":
            server = FedMTL(args, i)

        elif args.algorithm == "PerAvg":
            server = PerAvg(args, i)

        elif args.algorithm == "pFedMe":
            server = pFedMe(args, i)

        elif args.algorithm == "FedProx":
            server = FedProx(args, i)

        elif args.algorithm == "FedFomo":
            server = FedFomo(args, i)

        elif args.algorithm == "FedAMP":
            server = FedAMP(args, i)

        elif args.algorithm == "APFL":
            server = APFL(args, i)

        elif args.algorithm == "FedPer":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedPer(args, i)

        elif args.algorithm == "Ditto":
            server = Ditto(args, i)

        elif args.algorithm == "FedRep":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedRep(args, i)

        elif args.algorithm == "FedPHP":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedPHP(args, i)

        elif args.algorithm == "FedBN":
            server = FedBN(args, i)

        elif args.algorithm == "FedROD":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedROD(args, i)

        elif args.algorithm == "FedProto":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedProto(args, i)

        elif args.algorithm == "FedDyn":
            server = FedDyn(args, i)

        elif args.algorithm == "MOON":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = MOON(args, i)

        elif args.algorithm == "FedBABU":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedBABU(args, i)

        elif args.algorithm == "APPLE":
            server = APPLE(args, i)

        elif args.algorithm == "FedGen":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedGen(args, i)

        elif args.algorithm == "SCAFFOLD":
            server = SCAFFOLD(args, i)

        elif args.algorithm == "FedDistill":
            server = FedDistill(args, i)

        elif args.algorithm == "FedALA":
            server = FedALA(args, i)

        elif args.algorithm == "FedAWA":
            server = FedAWA(args, i)

        elif args.algorithm == "FedPAC":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedPAC(args, i)

        elif args.algorithm == "LG-FedAvg":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = LG_FedAvg(args, i)

        elif args.algorithm == "FedGC":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedGC(args, i)

        elif args.algorithm == "FML":
            server = FML(args, i)

        elif args.algorithm == "FedKD":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedKD(args, i)

        elif args.algorithm == "FedPCL":
            args.model.fc = nn.Identity()
            server = FedPCL(args, i)

        elif args.algorithm == "FedCP":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedCP(args, i)

        elif args.algorithm == "GPFL":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = GPFL(args, i)

        elif args.algorithm == "FedNTD":
            server = FedNTD(args, i)

        elif args.algorithm == "FedGH":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedGH(args, i)

        elif args.algorithm == "FedAvgDBE":
            args.head = copy.deepcopy(args.model.fc)
            args.model.fc = nn.Identity()
            args.model = BaseHeadSplit(args.model, args.head)
            server = FedAvgDBE(args, i)

        else:
            raise NotImplementedError
        print("\n=== Memory usage BEFORE training ===")
        process = psutil.Process(os.getpid())
        ram_before = process.memory_info().rss / (1024 ** 2)
        log_both(f"CPU RAM: {ram_before:.2f} MB")
        if args.device == "cuda":
            cuda_summary = torch.cuda.memory_summary()
        log_both(cuda_summary)
        server.train()
        print("\n=== Memory usage AFTER training ===")
        process = psutil.Process(os.getpid())
        ram_after = process.memory_info().rss / (1024 ** 2)
        log_both(f"CPU RAM: {ram_after:.2f} MB")
        if args.device == "cuda":
           cuda_summary = torch.cuda.memory_summary()
        log_both(cuda_summary)
        time_list.append(time.time()-start)

    print(f"\nAverage time cost: {round(np.average(time_list), 2)}s.")

    # Global average

    average_data(dataset=args.dataset, algorithm=args.algorithm, goal=args.goal, times=args.times)
    log_both(str(args.model))
    print_model_size(args.model)
    save_and_check_model_size(args.model)
    print("hello")
    print(f"[DEBUG] args.model is: {args.model}")
    # FLOPs Calculation
    from fvcore.nn import FlopCountAnalysis

    #print(f"[DEBUG] args.model is: {args.model}")

# FLOPs Calculation
    if isinstance(args.model, MultiHead_Seq2SeqLSTM):  # Check if it's an instance of MultiHead_Seq2SeqLSTM
            dummy_input = torch.randn(64, 10, 23).to(args.device)
            print("Entered MultiHead_Seq2SeqLSTM block")
    elif isinstance(args.model, Informer):  # Check if it's an instance of Informer
            dummy_input = torch.randn(64, 10, 23).to(args.device)
            print("Entered Informer block")

    elif isinstance(args.model, (cnn, resnet, alexnet, googlenet, mobilenet_v2)):  # Adjust if needed for other models
            dummy_input = torch.randn(1, 3, 32, 32).to(args.device)
    else:
        dummy_input = None
        print("No FLOPs calculation for this model type yet.")

    if dummy_input is not None:
        flops = FlopCountAnalysis(args.model, dummy_input)
        log_both(f"FLOPs: {flops.total() / 1e9:.2f} GFLOPs")


    log_both("All done!")

    reporter.report()


if __name__ == "__main__":
    total_start = time.time()

    parser = argparse.ArgumentParser()
    # general
    parser.add_argument('-go', "--goal", type=str, default="test",
                        help="The goal for this experiment")
    parser.add_argument('-dev', "--device", type=str, default="cuda",
                        choices=["cpu", "cuda"])
    parser.add_argument('-did', "--device_id", type=str, default="0")
    parser.add_argument('-data', "--dataset", type=str, default="driving10_fg34")
    parser.add_argument('-nb', "--num_classes", type=int, default=10)
    parser.add_argument('-m', "--model", type=str, default="informer")
    parser.add_argument('-lbs', "--batch_size", type=int, default=64)
    parser.add_argument('-lr', "--local_learning_rate", type=float, default=0.005,
                        help="Local learning rate")
    parser.add_argument('-ld', "--learning_rate_decay", type=bool, default=False)
    parser.add_argument('-ldg', "--learning_rate_decay_gamma", type=float, default=0.98)
    parser.add_argument('-gr', "--global_rounds", type=int, default=1)
    parser.add_argument('-ls', "--local_epochs", type=int, default=1,
                        help="Multiple update steps in one local epoch.")
    parser.add_argument('-algo', "--algorithm", type=str, default="FedAWA")
    parser.add_argument('-jr', "--join_ratio", type=float, default=1.0,
                        help="Ratio of clients per round")
    parser.add_argument('-rjr', "--random_join_ratio", type=bool, default=False,
                        help="Random ratio of clients per round")
    parser.add_argument('-nc', "--num_clients", type=int, default=10,
                        help="Total number of clients")
    parser.add_argument('-pv', "--prev", type=int, default=0,
                        help="Previous Running times")
    parser.add_argument('-t', "--times", type=int, default=1,
                        help="Running times")
    parser.add_argument('-eg', "--eval_gap", type=int, default=1,
                        help="Rounds gap for evaluation")
    parser.add_argument('-dp', "--privacy", type=bool, default=False,
                        help="differential privacy")
    parser.add_argument('-dps', "--dp_sigma", type=float, default=0.0)
    parser.add_argument('-sfn', "--save_folder_name", type=str, default='items')
    parser.add_argument('-ab', "--auto_break", type=bool, default=False)
    parser.add_argument('-dlg', "--dlg_eval", type=bool, default=False)
    parser.add_argument('-dlgg', "--dlg_gap", type=int, default=100)
    parser.add_argument('-bnpc', "--batch_num_per_client", type=int, default=2)
    parser.add_argument('-nnc', "--num_new_clients", type=int, default=0)
    parser.add_argument('-ften', "--fine_tuning_epoch_new", type=int, default=0)
    # practical
    parser.add_argument('-cdr', "--client_drop_rate", type=float, default=0.0,
                        help="Rate for clients that train but drop out")
    parser.add_argument('-tsr', "--train_slow_rate", type=float, default=0.0,
                        help="The rate for slow clients when training locally")
    parser.add_argument('-ssr', "--send_slow_rate", type=float, default=0.0,
                        help="The rate for slow clients when sending global model")
    parser.add_argument('-ts', "--time_select", type=bool, default=False,
                        help="Whether to group and select clients at each round according to time cost")
    parser.add_argument('-tth', "--time_threthold", type=float, default=10000,
                        help="The threthold for droping slow clients")
    # pFedMe / PerAvg / FedProx / FedAMP / FedPHP / GPFL
    parser.add_argument('-bt', "--beta", type=float, default=0.0)
    parser.add_argument('-lam', "--lamda", type=float, default=1.0,
                        help="Regularization weight")
    parser.add_argument('-mu', "--mu", type=float, default=0.0)
    parser.add_argument('-K', "--K", type=int, default=5,
                        help="Number of personalized training steps for pFedMe")
    parser.add_argument('-lrp', "--p_learning_rate", type=float, default=0.01,
                        help="personalized learning rate to caculate theta aproximately using K steps")
    # FedFomo
    parser.add_argument('-M', "--M", type=int, default=5,
                        help="Server only sends M client models to one client at each round")
    # FedMTL
    parser.add_argument('-itk', "--itk", type=int, default=4000,
                        help="The iterations for solving quadratic subproblems")
    # FedAMP
    parser.add_argument('-alk', "--alphaK", type=float, default=1.0,
                        help="lambda/sqrt(GLOABL-ITRATION) according to the paper")
    parser.add_argument('-sg', "--sigma", type=float, default=1.0)
    # APFL
    parser.add_argument('-al', "--alpha", type=float, default=1.0)
    # Ditto / FedRep
    parser.add_argument('-pls', "--plocal_epochs", type=int, default=1)
    # MOON
    parser.add_argument('-tau', "--tau", type=float, default=1.0)

    # FedBABU
    parser.add_argument('-fte', "--fine_tuning_epochs", type=int, default=10)
    # APPLE
    parser.add_argument('-dlr', "--dr_learning_rate", type=float, default=0.0)
    parser.add_argument('-L', "--L", type=float, default=1.0)
    # FedGen
    parser.add_argument('-nd', "--noise_dim", type=int, default=512)
    parser.add_argument('-glr', "--generator_learning_rate", type=float, default=0.005)
    parser.add_argument('-hd', "--hidden_dim", type=int, default=512)
    parser.add_argument('-se', "--server_epochs", type=int, default=1000)
    parser.add_argument('-lf', "--localize_feature_extractor", type=bool, default=False)
    # SCAFFOLD / FedGH
    parser.add_argument('-slr', "--server_learning_rate", type=float, default=1.0)
    # FedALA
    parser.add_argument('-et', "--eta", type=float, default=1.0)
    parser.add_argument('-s', "--rand_percent", type=int, default=80)

    # FedAWA/FedALA
    parser.add_argument('-p', "--layer_idx", type=int, default=2,
                        help="More fine-graind than its original paper.")
    # FedAWA
    parser.add_argument('-r', "--rounds", type=int, default=20)
    parser.add_argument('-pow', "--pow", type=float, default=1.0)
    # FedKD
    parser.add_argument('-mlr', "--mentee_learning_rate", type=float, default=0.005)
    parser.add_argument('-Ts', "--T_start", type=float, default=0.95)
    parser.add_argument('-Te', "--T_end", type=float, default=0.98)
    # FedAvgDBE
    parser.add_argument('-mo', "--momentum", type=float, default=0.1)
    parser.add_argument('-klw', "--kl_weight", type=float, default=0.0)


    args = parser.parse_args()

    os.environ["CUDA_VISIBLE_DEVICES"] = args.device_id

    if args.device == "cuda" and not torch.cuda.is_available():
        print("\ncuda is not avaiable.\n")
        args.device = "cpu"

    log_both("=" * 50)

    log_both("Algorithm: {}".format(args.algorithm))
    log_both("Local batch size: {}".format(args.batch_size))
    log_both("Local epochs: {}".format(args.local_epochs))
    log_both("Local learing rate: {}".format(args.local_learning_rate))
    log_both("Local learing rate decay: {}".format(args.learning_rate_decay))
    if args.learning_rate_decay:
        print("Local learing rate decay gamma: {}".format(args.learning_rate_decay_gamma))
    log_both("Total number of clients: {}".format(args.num_clients))
    log_both("Clients join in each round: {}".format(args.join_ratio))
    log_both("Clients randomly join: {}".format(args.random_join_ratio))
    log_both("Client drop rate: {}".format(args.client_drop_rate))
    log_both("Client select regarding time: {}".format(args.time_select))
    if args.time_select:
        print("Time threthold: {}".format(args.time_threthold))
    log_both("Running times: {}".format(args.times))
    log_both("Dataset: {}".format(args.dataset))
    log_both("Number of classes: {}".format(args.num_classes))
    log_both("Backbone: {}".format(args.model))
    log_both("Using device: {}".format(args.device))
    log_both("Using DP: {}".format(args.privacy))
    if args.privacy:
        print("Sigma for DP: {}".format(args.dp_sigma))
    log_both("Auto break: {}".format(args.auto_break))
    if not args.auto_break:
        log_both("Global rounds: {}".format(args.global_rounds))
    if args.device == "cuda":
        log_both("Cuda device id: {}".format(os.environ["CUDA_VISIBLE_DEVICES"]))
    log_both("DLG attack: {}".format(args.dlg_eval))
    if args.dlg_eval:
        print("DLG attack round gap: {}".format(args.dlg_gap))
    log_both("Total number of new clients: {}".format(args.num_new_clients))
    log_both("Fine tuning epoches on new clients: {}".format(args.fine_tuning_epoch_new))
    log_both("=" * 50)


    # if args.dataset == "mnist" or args.dataset == "fmnist":
    #     generate_mnist('../dataset/mnist/', args.num_clients, 10, args.niid)
    # elif args.dataset == "Cifar10" or args.dataset == "Cifar100":
    #     generate_cifar10('../dataset/Cifar10/', args.num_clients, 10, args.niid)
    # else:
    #     generate_synthetic('../dataset/synthetic/', args.num_clients, 10, args.niid)

    # with torch.profiler.profile(
    #     activities=[
    #         torch.profiler.ProfilerActivity.CPU,
    #         torch.profiler.ProfilerActivity.CUDA],
    #     profile_memory=True,
    #     on_trace_ready=torch.profiler.tensorboard_trace_handler('./log')
    #     ) as prof:
    # with torch.autograd.profiler.profile(profile_memory=True) as prof:
    def print_model_size(model):
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        log_both(f"Total parameters: {total_params:,}")
        log_both(f"Trainable parameters: {trainable_params:,}")

    def save_and_check_model_size(model, save_path="temp_model.pt"):
    # Save the model to disk
        torch.save(model.state_dict(), save_path)

    # Get file size in MB
        size_mb = os.path.getsize(save_path) / (1024 * 1024)
        print(f"Model size on disk: {size_mb:.2f} MB")
        log_both(f"Model size on disk: {size_mb:.2f} MB")

    # Optionally remove the file if it's temporary
        os.remove(save_path)
    run(args, log_both)

    log_both("=" * 50)
    log_both("Algorithm: {}".format(args.algorithm))
    log_both("Local epochs: {}".format(args.local_epochs))
    if not args.auto_break:
        log_both("Global rounds: {}".format(args.global_rounds))
    log_both("Dataset: {}".format(args.dataset))
    log_both("Running times: {}".format(args.times))
    log_both("Clients join in each round: {}".format(args.join_ratio))
    log_both("Clients randomly join: {}".format(args.random_join_ratio))
    log_both("=" * 50)
    # print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))
    # print(f"\nTotal time cost: {round(time.time()-total_start, 2)}s.")

summary_log.close()


Algorithm: FedAWA
Local batch size: 64
Local epochs: 1
Local learing rate: 0.005
Local learing rate decay: False
Total number of clients: 10
Clients join in each round: 1.0
Clients randomly join: False
Client drop rate: 0.0
Client select regarding time: False
Running times: 1
Dataset: driving10_fg34
Number of classes: 10
Backbone: informer
Using device: cuda
Using DP: False
Auto break: False
Global rounds: 1
Cuda device id: 0
DLG attack: False
Total number of new clients: 0
Fine tuning epoches on new clients: 0

============= Running time: 0th =============
Creating server and clients ...
Informer(
  (enc_embedding): DataEmbedding(
    (value_embedding): Linear(in_features=23, out_features=128, bias=True)
    (position_embedding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): Sequential(
    (0): InformerLayer(
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in